[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/solutions/42_topk_gather_solution.ipynb)

# 🟡 Solution: Top-k Gather

**Primitive: `topk` + `gather` with index expansion**

**Reduction:** `output[b, i]` = `values[b, j]` where `j` is the index of the `i`-th largest score in row `b`.

The key shape manipulation:
1. `torch.topk(scores, k, dim=-1).indices` → `(B, k)` — which columns to take
2. `unsqueeze(-1).expand(-1, -1, D)` → `(B, k, D)` — broadcast the column index across the feature dimension
3. `values.gather(dim=1, index)` → `(B, k, D)` — select

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def topk_gather(scores: torch.Tensor, values: torch.Tensor, k: int) -> torch.Tensor:
    # primitive: topk returns (B, k) indices; expand to (B, k, D) for gather
    _, idx = torch.topk(scores, k, dim=-1)                    # (B, k)
    idx = idx.unsqueeze(-1).expand(-1, -1, values.shape[-1])  # (B, k, D)
    return values.gather(dim=1, index=idx)                    # (B, k, D)

In [ ]:
# Verify
scores = torch.tensor([[0.1, 0.9, 0.4, 0.7]])
values = torch.tensor([[[1.,1.],[2.,2.],[3.,3.],[4.,4.]]])
result = topk_gather(scores, values, k=2)
print('output:', result.tolist())
print('expect: [[[2.0, 2.0], [4.0, 4.0]]]')
print('shape: ', result.shape)

In [ ]:
# Run judge
from torch_judge import check
check("topk_gather")